# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rishipatel092005/Fly-Rank-/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My Rule and Its Reason Codes

I will rank content pages by their observed refresh opportunity using three pre-decision signals: search volume, CTR, and average search position.

The rule gives higher priority to pages that have meaningful search demand and appear to have an opportunity to improve their search performance. Higher search volume increases the potential value of a refresh, while CTR and average position help identify pages where performance may be weaker than the available search opportunity.

The rule produces one primary reason code for each page:

- HIGH_VOLUME_OPPORTUNITY — meaningful search demand with an observable opportunity for improvement.
- CTR_OPPORTUNITY — relatively low CTR compared with the page's search position.
- POSITION_OPPORTUNITY — the page has meaningful search demand but is ranking below the strongest positions.

The output also contains an action label. High-scoring pages are labelled REVIEW because they should be considered first by the content/SEO team.

This is a directional decision-support baseline, not a causal model. A high score does not guarantee that refreshing a page will improve its performance.

## Signal Check 1 — Content Freshness

I expect content freshness to be relevant to refresh prioritization because older or less recently updated content may have a stronger reason to be reviewed.

I will compare the observed decline rate across freshness buckets and use the result to decide whether freshness is a useful signal for the baseline.

The verdict will be based on the observed data rather than an assumed relationship.

In [9]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the observed outcome used only for this signal audit
# Do NOT use this column in the final baseline score.
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Check which freshness-related fields are available
freshness_candidates = [
    col for col in df.columns
    if any(word in col.lower() for word in ["fresh", "update", "age"])
]

print("Available freshness-related columns:")
print(freshness_candidates)

# Use the freshness bucket if it exists
if "freshness_tier" in df.columns:

    freshness_check = (
        df.groupby("freshness_tier", dropna=False)
          .agg(
              n=("content_id", "size"),
              declining_rate=("is_declining", "mean")
          )
          .reset_index()
    )

    freshness_check["declining_rate"] = (
        freshness_check["declining_rate"] * 100
    ).round(2)

    display(freshness_check)

else:
    print(
        "freshness_tier is not available. "
        "Use one of the available freshness-related columns shown above."
    )

Available freshness-related columns:
['pageviews_90d', 'engaged_sessions_90d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'engagement_rate']


,freshness_tier,n,declining_rate
0,0-30,20480,51.14
1,181+,174,47.13
2,31-90,175,58.86
3,91-180,9171,61.11


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Keep only information available at decision time.
# Do NOT use trend_direction or trend_pct because they are label-derived.
work = df.copy()

# Make sure required numeric columns are numeric
numeric_cols = [
    "search_volume",
    "ctr",
    "avg_position"
]

for col in numeric_cols:
    work[col] = pd.to_numeric(work[col], errors="coerce")

# Fill missing values with dataset medians
for col in numeric_cols:
    work[col] = work[col].fillna(work[col].median())

# Normalize signals to 0-1 using percentile ranks.
# Percentile ranking makes the rule less sensitive to extreme values.
work["volume_score"] = work["search_volume"].rank(pct=True)

# Lower CTR means more opportunity, so reverse the percentile rank.
work["ctr_opportunity"] = 1 - work["ctr"].rank(pct=True)

# Higher average position number means worse ranking position,
# so higher values represent more potential opportunity.
work["position_opportunity"] = work["avg_position"].rank(pct=True)

# Baseline score
work["baseline_score"] = (
    0.40 * work["volume_score"]
    + 0.30 * work["ctr_opportunity"]
    + 0.30 * work["position_opportunity"]
)

# Assign ONE primary reason code
def get_reason(row):
    signals = {
        "HIGH_VOLUME_OPPORTUNITY": row["volume_score"],
        "CTR_OPPORTUNITY": row["ctr_opportunity"],
        "POSITION_OPPORTUNITY": row["position_opportunity"]
    }
    return max(signals, key=signals.get)

work["reason_code"] = work.apply(get_reason, axis=1)

# Action label
work["action"] = np.where(
    work["baseline_score"] >= work["baseline_score"].quantile(0.80),
    "REVIEW",
    "MONITOR"
)

# Rank all pages
work = work.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)

# Create the required output
output_cols = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]

queue = work[output_cols].copy()

# Write the generated queue
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print("Rows ranked:", len(queue))
print("Output written to:", output_path)
display(queue.head(20))

Rows ranked: 30000
Output written to: work/outputs/baseline_action_score.csv


,rank,content_id,baseline_score,reason_code,action
0,1,content_3aa75aa83ed3,0.930315,HIGH_VOLUME_OPPORTUNITY,REVIEW
1,2,content_3921a7d7066d,0.930135,HIGH_VOLUME_OPPORTUNITY,REVIEW
2,3,content_661e1745db72,0.929615,POSITION_OPPORTUNITY,REVIEW
3,4,content_83e3da1394ac,0.929223,HIGH_VOLUME_OPPORTUNITY,REVIEW
4,5,content_2ea49fcfdf36,0.929028,POSITION_OPPORTUNITY,REVIEW
5,6,content_e34fe8c07fa2,0.928102,HIGH_VOLUME_OPPORTUNITY,REVIEW
6,7,content_a11501e6e45d,0.927665,HIGH_VOLUME_OPPORTUNITY,REVIEW
7,8,content_c41622fe1359,0.925973,POSITION_OPPORTUNITY,REVIEW
8,9,content_fa5ac489c35f,0.925790,HIGH_VOLUME_OPPORTUNITY,REVIEW
9,10,content_4983b29e8a51,0.925403,POSITION_OPPORTUNITY,REVIEW


## 2. Build the Ranked Queue

The baseline combines three signals that are available at decision time:

- `search_volume` — represents the potential search demand for a page.
- `ctr` — represents observed click-through performance.
- `avg_position` — represents the page's average search visibility.

Each signal is converted to a percentile-based score so that the three signals can be combined despite having different scales.

The final baseline score is:

`0.40 × volume_score + 0.30 × ctr_opportunity + 0.30 × position_opportunity`

The pages are ranked from highest to lowest baseline score.

Each page receives exactly one primary reason code and one action label. Pages in the top 20% of the score distribution are labelled `REVIEW`; the remaining pages are labelled `MONITOR`.

This is a simple, interpretable baseline. It is intended to prioritize pages for human review rather than automatically determine that a page should be refreshed.

## 3. Top-20 Review

I reviewed the 20 highest-ranked pages as a skeptic rather than assuming that every high score represents a true refresh opportunity.

For each page, I considered:

- the action recommended by the rule,
- the primary reason code,
- the strength of the observed signals,
- my confidence in the recommendation,
- and what could make the recommendation wrong.

The baseline is intended to prioritize pages for human review, not automatically decide that every selected page should be refreshed.

A high ranking can still be a weak recommendation if the underlying signal has another explanation, such as search intent, SERP characteristics, or naturally low CTR for the page's query mix.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
top20 = queue.head(20).copy()

display(top20)

,rank,content_id,baseline_score,reason_code,action
0,1,content_3aa75aa83ed3,0.930315,HIGH_VOLUME_OPPORTUNITY,REVIEW
1,2,content_3921a7d7066d,0.930135,HIGH_VOLUME_OPPORTUNITY,REVIEW
2,3,content_661e1745db72,0.929615,POSITION_OPPORTUNITY,REVIEW
3,4,content_83e3da1394ac,0.929223,HIGH_VOLUME_OPPORTUNITY,REVIEW
4,5,content_2ea49fcfdf36,0.929028,POSITION_OPPORTUNITY,REVIEW
5,6,content_e34fe8c07fa2,0.928102,HIGH_VOLUME_OPPORTUNITY,REVIEW
6,7,content_a11501e6e45d,0.927665,HIGH_VOLUME_OPPORTUNITY,REVIEW
7,8,content_c41622fe1359,0.925973,POSITION_OPPORTUNITY,REVIEW
8,9,content_fa5ac489c35f,0.925790,HIGH_VOLUME_OPPORTUNITY,REVIEW
9,10,content_4983b29e8a51,0.925403,POSITION_OPPORTUNITY,REVIEW


In [15]:
# Create a review table for the top 20 recommendations.
top20_review = top20.copy()

top20_review["confidence_note"] = (
    "Moderate — recommendation is supported by the observed baseline signals."
)

top20_review["what_would_make_it_wrong"] = (
    "The observed signal may have another explanation, "
    "so a high score does not prove that refreshing the page will improve performance."
)

top20_review = top20_review[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,rank,content_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_3aa75aa83ed3,0.930315,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
1,2,content_3921a7d7066d,0.930135,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
2,3,content_661e1745db72,0.929615,POSITION_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
3,4,content_83e3da1394ac,0.929223,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
4,5,content_2ea49fcfdf36,0.929028,POSITION_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
5,6,content_e34fe8c07fa2,0.928102,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
6,7,content_a11501e6e45d,0.927665,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
7,8,content_c41622fe1359,0.925973,POSITION_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
8,9,content_fa5ac489c35f,0.925790,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...
9,10,content_4983b29e8a51,0.925403,POSITION_OPPORTUNITY,REVIEW,Moderate — recommendation is supported by the ...,The observed signal may have another explanati...


In [16]:
top20_review = top20.copy()

def confidence_note(reason):
    if reason == "HIGH_VOLUME_OPPORTUNITY":
        return (
            "Moderate — high search demand supports review priority, "
            "but demand alone does not prove a refresh is needed."
        )
    elif reason == "CTR_OPPORTUNITY":
        return (
            "Moderate — relatively low CTR suggests an opportunity, "
            "but CTR can vary with query intent and SERP features."
        )
    elif reason == "POSITION_OPPORTUNITY":
        return (
            "Moderate — weaker average position supports review, "
            "but position alone does not prove content quality is the issue."
        )
    return "Low to moderate — requires human review."

def wrong_reason(reason):
    if reason == "HIGH_VOLUME_OPPORTUNITY":
        return (
            "The page may already satisfy search intent despite having "
            "high search demand."
        )
    elif reason == "CTR_OPPORTUNITY":
        return (
            "Low CTR may be caused by query mix or SERP characteristics "
            "rather than weak content."
        )
    elif reason == "POSITION_OPPORTUNITY":
        return (
            "The ranking position may be appropriate for the query, "
            "so a content refresh may not improve it."
        )
    return "The observed signal may have another explanation."

top20_review["confidence_note"] = top20_review["reason_code"].apply(
    confidence_note
)

top20_review["what_would_make_it_wrong"] = top20_review["reason_code"].apply(
    wrong_reason
)

top20_review = top20_review[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,rank,content_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_3aa75aa83ed3,0.930315,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — high search demand supports review ...,The page may already satisfy search intent des...
1,2,content_3921a7d7066d,0.930135,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — high search demand supports review ...,The page may already satisfy search intent des...
2,3,content_661e1745db72,0.929615,POSITION_OPPORTUNITY,REVIEW,Moderate — weaker average position supports re...,The ranking position may be appropriate for th...
3,4,content_83e3da1394ac,0.929223,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — high search demand supports review ...,The page may already satisfy search intent des...
4,5,content_2ea49fcfdf36,0.929028,POSITION_OPPORTUNITY,REVIEW,Moderate — weaker average position supports re...,The ranking position may be appropriate for th...
5,6,content_e34fe8c07fa2,0.928102,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — high search demand supports review ...,The page may already satisfy search intent des...
6,7,content_a11501e6e45d,0.927665,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — high search demand supports review ...,The page may already satisfy search intent des...
7,8,content_c41622fe1359,0.925973,POSITION_OPPORTUNITY,REVIEW,Moderate — weaker average position supports re...,The ranking position may be appropriate for th...
8,9,content_fa5ac489c35f,0.925790,HIGH_VOLUME_OPPORTUNITY,REVIEW,Moderate — high search demand supports review ...,The page may already satisfy search intent des...
9,10,content_4983b29e8a51,0.925403,POSITION_OPPORTUNITY,REVIEW,Moderate — weaker average position supports re...,The ranking position may be appropriate for th...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak Picks + Leakage Check

Some high-ranked pages may be weak recommendations even when their baseline score is high. For example, a page may have high search volume but still be performing appropriately for its search intent. A low CTR can also be explained by SERP features or differences in query intent rather than poor content quality.

The baseline should therefore be treated as a prioritization tool for human review, not as proof that a page needs a refresh.

### Leakage Check

The baseline uses only information available at the decision moment: search volume, CTR, and average position.

I deliberately excluded trend_direction and trend_pct because they are label-derived/future-outcome information and could leak the outcome into the baseline.

No future-window information or product flags are used in the score.

In [ ]:
# Explicitly verify that leakage-prone columns are not used
used_features = [
    "search_volume",
    "ctr",
    "avg_position"
]

leakage_columns = [
    "trend_direction",
    "trend_pct"
]

print("Features used:", used_features)
print("Leakage-prone columns excluded:", leakage_columns)

assert "trend_direction" not in used_features
assert "trend_pct" not in used_features

print("Leakage check passed.")

Features used: ['search_volume', 'ctr', 'avg_position']
Leakage-prone columns excluded: ['trend_direction', 'trend_pct']
Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.